In [1]:
import numpy as np 
import pandas as pd 
import time
import random 


In [2]:
# Open .txt file using appropriate delimiter
df_data = pd.read_csv('dataForAssignment4.txt', delimiter=',', header=None)

# Clean up the extra spaces and then split the columns
df_data_clean = df_data[0].str.strip().replace(r'\s+', ' ', regex=True)

# SPlit the columns using a single space as the delimiter
df_data_split = df_data_clean.str.split(' ', expand=True)

# Convert to numeric, coercing any non-convertible values to NaN
df_data_numeric = df_data_split.apply(pd.to_numeric, errors='coerce')

In [27]:
print(df_data_numeric.head())

          0           1           2           3           4           5    \
0  127.444480  145.609520  148.774380  147.251710  155.363900  162.706100   
1    1.631022   11.335683   11.773843   12.096300    7.915325   19.015742   
2   12.258457   14.139799   17.539636    9.726507   18.812856   12.553163   
3   -0.275116    2.071711    6.694510   13.619866   11.759128   25.035415   
4    9.155447   -0.466132    8.455586    9.622885   19.659208   11.717348   

          6           7           8           9    ...        118        119  \
0  179.497860  179.712750  166.279920  186.450040  ...  75.839724  75.743199   
1   17.127280   15.713202   18.143192   18.102330  ...  -9.135946  -4.919573   
2   18.105615   20.843602   19.891813   23.218734  ...  16.335423  11.379510   
3   26.129791   33.329513   35.469797   40.143830  ... -42.876570 -38.915197   
4   11.179688   15.335171   16.523433   16.931238  ... -11.898610 -11.313026   

         120        121        122         123         1

Function that includes both pruning steps, but can be implemented with or without them

In [ ]:
def euclidean_distance(row1, row2):
    return np.sqrt(np.sum((row1 - row2) ** 2))


def find_outlying_object(data, prune_step_1=True, prune_step_2=True):

    n = len(data)
    best_so_far = 0
    best_index = -1

    # randomly select a reference object
    reference_index = np.random.randint(0, n)
    reference_object = data[reference_index]

    # Precompute referenced distances
    referenced_distances = np.array([euclidean_distance(reference_object, data[i]) for i in range(n)])

    for query_index in range(n):
        current_best = float("inf")

        for i in range(n):
            if i == query_index:
                continue
            # Pruning Step 2
            if prune_step_2 and referenced_distances[query_index] + referenced_distances[i] < best_so_far:
                current_best = 0
                break
            
            d = euclidean_distance(data[query_index], data[i])

            # Pruning Step 1
            if prune_step_1 and d < best_so_far:
                current_best = 0
                break

            # Update current best
            if d < current_best:
                current_best = d

        if current_best > best_so_far:
            best_so_far = current_best
            best_index = query_index

    return best_so_far, best_index


In [19]:
# Z-normalize across ROWS
df_norm_data = df_data_numeric.apply(lambda x: (x - x.mean()) / (x.std()), axis=1) # axis = 1 indicates row
print(df_norm_data.shape)

(16000, 128)


Execution without pruning

In [24]:
start_time_no_pruning = time.time()

df_norm_data_np = df_norm_data.to_numpy()

best_distance, best_index = find_outlying_object(df_norm_data_np, prune_step_1=False, prune_step_2=False)
print(f"Best Distance: {best_distance}")
print(f"Best Index: {best_index}")

end_time_no_pruning = time.time()

total_no_pruning_time = end_time_no_pruning - start_time_no_pruning
print(f'Time it takes to find outliers using no pruning steps: {total_no_pruning_time/60} minutes')

Best Distance: 14.192666847149797
Best Index: 10523
Time it takes to find outliers using no pruning steps: 31.237238283952077 minutes


Execution with pruning step 1

In [25]:
start_time_pruning_1 = time.time()

df_norm_data_np = df_norm_data.to_numpy()

best_distance, best_index = find_outlying_object(df_norm_data_np, prune_step_1=True, prune_step_2=False)
print(f"Best Distance: {best_distance}")
print(f"Best Index: {best_index}")

end_time_pruning_1 = time.time()

total_pruning_1_time = end_time_pruning_1 - start_time_pruning_1
print(f'Time it takes to find outliers using pruning step 1: {total_pruning_1_time/60} minutes')

Best Distance: 14.192666847149797
Best Index: 10523
Time it takes to find outliers using pruning step 1: 5.471809434890747 minutes


Execution with pruning step 2

In [26]:
start_time_pruning_2 = time.time()

df_norm_data_np = df_norm_data.to_numpy()

best_distance, best_index = find_outlying_object(df_norm_data_np, prune_step_1=False, prune_step_2=True)
print(f"Best Distance: {best_distance}")
print(f"Best Index: {best_index}")

end_time_pruning_2 = time.time()

total_pruning_2_time = end_time_pruning_2 - start_time_pruning_2
print(f'Time it takes to find outliers using pruning step 2: {total_pruning_2_time/60} minutes')

Best Distance: 14.192666847149797
Best Index: 10523
Time it takes to find outliers using pruning step 2: 26.613878552118937 minutes


Execution with both pruning steps

In [23]:
start_time_all_pruning = time.time()

df_norm_data_np = df_norm_data.to_numpy()

best_distance, best_index = find_outlying_object(df_norm_data_np, prune_step_1=True, prune_step_2=True)
print(f"Best Distance: {best_distance}")
print(f"Best Index: {best_index}")

end_time_all_pruning = time.time()

total_all_pruning_time = end_time_all_pruning - start_time_all_pruning
print(f'Time it takes to find outliers using both pruning steps: {total_all_pruning_time/60} minutes')

Best Distance: 14.192666847149797
Best Index: 10523
Time it takes to find outliers using both pruning steps: 5.746860961119334 minutes
